# Phase B13 — `r009_liquidity_sentiment`: 5-Seed Ensemble with Liquidity + FinBERT Sentiment

**Setup:** Runtime → GPU (T4 or better).

**What's different from r008:**
- All r008 flags on (`use_liquidity_features: True`, `liquidity_noise_enabled: True`).
- `use_sentiment_features: True` — adds 3 global features per timestep:
  bullish_frac, neutral_frac, headline_count_norm from `data/sentiment_daily.parquet`.
  Obs width 75 → 78.
- `data/sentiment_daily.parquet` **must be included in your zip** (run
  `python scripts/fetch_sentiment.py --infer` locally first).
- All seeds / steps / Transformer dims identical to r007/r008 for clean ablation.

**Workflow:**
1. Cell 1: install deps (adds `transformers` for FinBERT verification).
2. Cell 2: upload `algo-trading-rl.zip` (must include `data/sentiment_daily.parquet`).
3. Cell 3: verify GPU.
4. Cell 4: verify sentiment file + patch `configs/transformer.yaml`.
5. Cell 5: train 5 seeds (~5–6 GPU-hrs on T4).
6. Cell 6: ensemble_eval.
7. Cell 7: download `colab_output_r009.zip`.


In [ ]:
!pip install -q gymnasium stable-baselines3 yfinance ta pyyaml


In [ ]:
import os, zipfile, sys
from google.colab import files

print('Select algo-trading-rl.zip when the dialog appears...')
uploaded = files.upload()
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('.')
for candidate in ('algo-trading-rl', 'algo-trading-rl-main'):
    if os.path.isdir(candidate):
        os.chdir(candidate)
        break
sys.path.insert(0, os.getcwd())
print('Working dir:', os.getcwd())
assert os.path.exists('scripts/train_multi_asset_transformer.py')
assert os.path.exists('scripts/ensemble_eval.py')
assert os.path.exists('envs/multi_asset_env.py')
assert os.path.exists('data/SPY.parquet'), 'data/*.parquet missing from zip'
assert os.path.exists('data/sentiment_daily.parquet'), \
    'data/sentiment_daily.parquet missing — run fetch_sentiment.py --infer locally first'


In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available(): print('Device:', torch.cuda.get_device_name(0))


In [ ]:
import yaml, pprint
import pandas as pd

# Verify sentiment file
sent = pd.read_parquet('data/sentiment_daily.parquet')
coverage = (sent['headline_count_norm'] > 0).mean() * 100
print(f'Sentiment: {len(sent)} trading days, {coverage:.1f}% real coverage (rest neutral fill)')

# Patch configs/transformer.yaml — all three Phase flags on
CFG_PATH = 'configs/transformer.yaml'
with open(CFG_PATH) as f:
    cfg = yaml.safe_load(f)

cfg['multi_asset']['use_liquidity_features']  = True
cfg['multi_asset']['liquidity_noise_enabled'] = True
cfg['multi_asset']['use_sentiment_features']  = True

with open(CFG_PATH, 'w') as f:
    yaml.dump(cfg, f, sort_keys=False)

print('\nmulti_asset block after patch:')
pprint.pprint(cfg['multi_asset'])


In [ ]:
import subprocess

SEEDS = [42, 101, 202, 303, 404]
for s in SEEDS:
    print(f'\n===== SEED {s} =====')
    rc = subprocess.call([
        'python', 'scripts/train_multi_asset_transformer.py',
        '--config', 'configs/transformer.yaml',
        '--tag', f'r009_liquidity_sentiment_seed{s}',
        '--seed', str(s),
        '--val-eval-freq', '10000',
    ])
    assert rc == 0, f'seed {s} training failed'


In [ ]:
import glob, shutil, os
ENS_DIR = 'runs/ensemble_r009_liquidity_sentiment'
os.makedirs(ENS_DIR, exist_ok=True)
for p in glob.glob('runs/*_r009_liquidity_sentiment_seed*'):
    dest = os.path.join(ENS_DIR, os.path.basename(p))
    if not os.path.exists(dest):
        shutil.move(p, dest)
print('Members:', sorted(os.listdir(ENS_DIR)))

!python scripts/ensemble_eval.py --run-dir {ENS_DIR} --config configs/transformer.yaml --scale 5.0


In [ ]:
import shutil, os
shutil.make_archive('colab_output_r009', 'zip', 'runs')
print(f'colab_output_r009.zip ({os.path.getsize("colab_output_r009.zip")/1024:.0f} KB)')
from google.colab import files
files.download('colab_output_r009.zip')
